# NN_10 — SVM e GMM-LRT nas 4 Features

Duas abordagens clássicas de ML aplicadas às 4 features [|τ|, ĥ, SNR, E]:

1. **SVM** (SGDClassifier com hinge loss): função de decisão ilimitada → D3F compatível
2. **GMM-LRT**: ajusta GMMs separadas para H₀ e H₁ → log-likelihood ratio como estatística de teste (diretamente ótimo por Neyman-Pearson)

Ambas evitam o problema de saturação do sigmoid.

In [ ]:
# ==============================================================================
# 1. IMPORTS E INSTALAÇÃO
# ==============================================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'], check=True)

import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from pathlib import Path
from scipy.stats import norm
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import gc
import warnings
warnings.filterwarnings('ignore')

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
results_dir  = project_root / 'results'
data_dir     = results_dir / 'data'
models_dir   = results_dir / 'models'
vis_dir      = results_dir / 'visualizations'

ALPHA    = 1e-7
Q_INV    = norm.ppf(1 - ALPHA)
SNR_BINS = [0, 5, 10, 15, 20, 25, 30]
print(f'scikit-learn pronto. α={ALPHA:.0e}  Q⁻¹={Q_INV:.4f}')

In [ ]:
# ==============================================================================
# 2. CARREGAR DATASET + EXTRAIR 4 FEATURES
# ==============================================================================
dataset_path = data_dir / 'dataset_cnn_yeq_0_30dB.h5'

with h5py.File(str(dataset_path), 'r') as f:
    Y_train   = f['train/y_eq'][:].astype(np.float32)
    y_train   = f['train/y'][:].astype(np.float32)
    snr_train = f['train/snr'][:]
    TAU_train = np.abs(f['train/tau_eq'][:]).astype(np.float32)

    Y_val     = f['val/y_eq'][:].astype(np.float32)
    y_val     = f['val/y'][:].astype(np.float32)
    snr_val   = f['val/snr'][:]
    TAU_val   = np.abs(f['val/tau_eq'][:]).astype(np.float32)

    Y_test    = f['test/y_eq'][:].astype(np.float32)
    y_test    = f['test/y'][:].astype(np.float32)
    snr_test  = f['test/snr'][:]
    TAU_test  = np.abs(f['test/tau_eq'][:]).astype(np.float32)

def extract_features(Y, TAU, snr):
    h_est = np.abs(Y).mean(axis=1)
    E     = (Y ** 2).mean(axis=1)
    return np.stack([TAU, h_est, snr.astype(np.float32), E], axis=1)

X_train = extract_features(Y_train, TAU_train, snr_train)
X_val   = extract_features(Y_val,   TAU_val,   snr_val)
X_test  = extract_features(Y_test,  TAU_test,  snr_test)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Distribuição treino  H0={int((y_train==0).sum())} H1={int((y_train==1).sum())}')

# Carregar resultados clássicos para referência
with open(str(data_dir / 'nn08_architecture_comparison.json')) as f:
    ref = json.load(f)
classical_pd = {int(k): v for k, v in ref['classical'].items()}

In [ ]:
# ==============================================================================
# 3. TREINAR SVM — SGDClassifier (escalável para 280k amostras)
# ==============================================================================
# SGD com hinge loss ≡ Linear SVM, mas escalável.
# Para capturar não-linearidade: adicionar RBF features via Nystroem (opcional)
from sklearn.kernel_approximation import Nystroem

print('Treinando SVM linear (SGD + hinge)...')
svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SGDClassifier(loss='hinge', alpha=1e-4, max_iter=100,
                             tol=1e-4, random_state=42, n_jobs=-1,
                             class_weight='balanced', verbose=0))
])
svm_linear.fit(X_train, y_train)
auc_svm_lin = roc_auc_score(y_val, svm_linear.decision_function(X_val))
print(f'  SVM Linear  val AUC = {auc_svm_lin:.5f}')

print('\nTreinando SVM com features RBF aproximadas (Nystroem)...')
svm_rbf = Pipeline([
    ('scaler',   StandardScaler()),
    ('nystroem', Nystroem(kernel='rbf', gamma=0.1, n_components=200, random_state=42)),
    ('svm',      SGDClassifier(loss='hinge', alpha=1e-4, max_iter=100,
                               tol=1e-4, random_state=42, n_jobs=-1,
                               class_weight='balanced', verbose=0))
])
svm_rbf.fit(X_train, y_train)
auc_svm_rbf = roc_auc_score(y_val, svm_rbf.decision_function(X_val))
print(f'  SVM RBF (Nystroem) val AUC = {auc_svm_rbf:.5f}')

# Usar o melhor
best_svm = svm_rbf if auc_svm_rbf >= auc_svm_lin else svm_linear
best_svm_name = 'SVM-RBF' if auc_svm_rbf >= auc_svm_lin else 'SVM-Linear'
print(f'\nMelhor SVM: {best_svm_name}')

In [ ]:
# ==============================================================================
# 4. TREINAR GMM-LRT
# ==============================================================================
# Ajusta GMMs separadas para H₀ e H₁.
# LLR = log p(x|H₁) - log p(x|H₀) é a estatística ótima de Neyman-Pearson.

print('Treinando GMMs...')
X_h0 = X_train[y_train == 0]
X_h1 = X_train[y_train == 1]

best_bic = np.inf
best_n   = 1
best_gmm_h0 = None
best_gmm_h1 = None

# Selecionar número de componentes por BIC
for n in [1, 2, 4, 8]:
    g0 = GaussianMixture(n_components=n, covariance_type='full',
                         random_state=42, max_iter=200)
    g1 = GaussianMixture(n_components=n, covariance_type='full',
                         random_state=42, max_iter=200)
    g0.fit(X_h0)
    g1.fit(X_h1)
    bic = g0.bic(X_h0) + g1.bic(X_h1)
    print(f'  GMM n={n}  BIC={bic:.1f}')
    if bic < best_bic:
        best_bic, best_n = bic, n
        best_gmm_h0, best_gmm_h1 = g0, g1

print(f'\nMelhor n_components = {best_n}  (BIC={best_bic:.1f})')

# Log-likelihood ratio no teste
llr_test = best_gmm_h1.score_samples(X_test) - best_gmm_h0.score_samples(X_test)
auc_gmm  = roc_auc_score(y_test, llr_test)
print(f'GMM-LRT test AUC = {auc_gmm:.5f}')

In [ ]:
# ==============================================================================
# 5. D3F SOBRE ESTATÍSTICAS SVM E GMM-LRT
# ==============================================================================
def d3f_threshold(scores_h0, alpha=1e-7):
    mu, sigma = scores_h0.mean(), scores_h0.std()
    return float(mu + norm.ppf(1 - alpha) * sigma), float(mu), float(sigma)

def eval_d3f(scores, y_true, snr, alpha=1e-7, clip=None):
    results = {}; thresholds = {}
    for snr_db in SNR_BINS:
        mask_h0 = (snr >= snr_db-2.5) & (snr < snr_db+2.5) & (y_true==0)
        mask_h1 = (snr >= snr_db-2.5) & (snr < snr_db+2.5) & (y_true==1)
        if mask_h0.sum() < 30 or mask_h1.sum() < 10:
            results[snr_db]=None; thresholds[snr_db]=None; continue
        thr, _, _ = d3f_threshold(scores[mask_h0], alpha)
        if clip: thr = np.clip(thr, clip[0], clip[1])
        results[snr_db]    = float((scores[mask_h1] > thr).mean())
        thresholds[snr_db] = thr
    return results, thresholds

# Escores SVM no teste
svm_scores_test = best_svm.decision_function(X_test)

# Escores GMM-LRT no teste
gmm_scores_test = llr_test

pd_svm, thr_svm = eval_d3f(svm_scores_test, y_test, snr_test)
pd_gmm, thr_gmm = eval_d3f(gmm_scores_test, y_test, snr_test)

print('SNR  | Classical | SVM-D3F  | GMM-LRT-D3F')
print('-' * 50)
for snr_db in SNR_BINS:
    cl = classical_pd.get(snr_db, 0)
    sv = pd_svm.get(snr_db) or 0
    gm = pd_gmm.get(snr_db) or 0
    print(f'{snr_db:3d}  | {cl:8.4f}  | {sv:7.4f}  | {gm:10.4f}')

In [ ]:
# ==============================================================================
# 6. ANÁLISE POR BIN DE SNR — GAUSSIANIDADE DOS ESCORES H₀
# ==============================================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Distribuição H₀ dos escores SVM e GMM-LRT por SNR', fontsize=12)

for i, snr_db in enumerate(SNR_BINS):
    ax = axes[i // 4][i % 4]
    mask_h0 = (snr_test >= snr_db-2.5) & (snr_test < snr_db+2.5) & (y_test==0)
    mask_h1 = (snr_test >= snr_db-2.5) & (snr_test < snr_db+2.5) & (y_test==1)
    if mask_h0.sum() < 10: ax.set_visible(False); continue

    # GMM scores
    s_h0 = gmm_scores_test[mask_h0]
    s_h1 = gmm_scores_test[mask_h1]
    ax.hist(s_h0, bins=40, density=True, alpha=0.5, color='tomato',   label='H₀')
    ax.hist(s_h1, bins=40, density=True, alpha=0.5, color='steelblue', label='H₁')
    if thr_gmm.get(snr_db):
        ax.axvline(thr_gmm[snr_db], color='k', ls='--', lw=1.5,
                   label=f'τ*={thr_gmm[snr_db]:.1f}')
    ax.set_title(f'GMM-LRT @ {snr_db} dB  PD={pd_gmm.get(snr_db) or 0:.3f}', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

# Último painel: PD vs SNR comparativo
ax = axes[1][3]
ax.plot(SNR_BINS, [classical_pd.get(s,0) for s in SNR_BINS], 'k-o', lw=2, label='Clássico')
ax.plot(SNR_BINS, [pd_svm.get(s) or 0 for s in SNR_BINS],   'm-s', lw=2, label=best_svm_name)
ax.plot(SNR_BINS, [pd_gmm.get(s) or 0 for s in SNR_BINS],   'g-^', lw=2, label='GMM-LRT')
ax.set(xlabel='SNR (dB)', ylabel='PD', title='PD vs SNR (α=10⁻⁷)',
       ylim=(-0.05,1.05)); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
vis_path = vis_dir / 'NN10_SVM_GMM_Results.png'
plt.savefig(str(vis_path), dpi=120)
plt.show()
print(f'Salvo → {vis_path}')

In [ ]:
# ==============================================================================
# 7. SALVAR RESULTADOS
# ==============================================================================
import joblib

# Salvar modelos
joblib.dump(best_svm,      str(models_dir / 'svm_4feat.pkl'))
joblib.dump(best_gmm_h0,   str(models_dir / 'gmm_h0_4feat.pkl'))
joblib.dump(best_gmm_h1,   str(models_dir / 'gmm_h1_4feat.pkl'))

results = {
    'alpha':           ALPHA,
    'svm_model':       best_svm_name,
    'svm_val_auc':     float(auc_svm_rbf if best_svm_name=='SVM-RBF' else auc_svm_lin),
    'gmm_n_components': best_n,
    'gmm_test_auc':    float(auc_gmm),
    'classical':       {str(k): v for k, v in classical_pd.items()},
    'svm_d3f':         {str(k): pd_svm.get(k) for k in SNR_BINS},
    'gmm_lrt_d3f':     {str(k): pd_gmm.get(k) for k in SNR_BINS},
    'thresholds': {
        'svm':     {str(k): thr_svm.get(k) for k in SNR_BINS},
        'gmm_lrt': {str(k): thr_gmm.get(k) for k in SNR_BINS},
    }
}

out_path = data_dir / 'nn10_svm_gmm.json'
with open(str(out_path), 'w') as f:
    json.dump(results, f, indent=2)
print(f'Resultados salvos → {out_path}')
print(f'\nMelhor PD SVM    @ 30dB: {pd_svm.get(30) or 0:.4f}')
print(f'Melhor PD GMM-LRT@ 30dB: {pd_gmm.get(30) or 0:.4f}')
print(f'Clássico         @ 10dB: {classical_pd.get(10, 0):.4f}')

In [ ]:
# ==============================================================================
# 8. CARREGAR z_eq (1024-dim) PARA COMPARAÇÃO SVM/GMM
# ==============================================================================
# z_eq = y/h - rho_s*msg  →  H1: rho_t*t + w/h  |  H0: ≈ w/h
# Usar z_eq como input bruto para SVM/GMM permite avaliar se
# as 4 features manuais já capturam toda a informação discriminativa.

with h5py.File(str(dataset_path), 'r') as f:
    if 'train/z_eq' not in f:
        raise KeyError(
            "z_eq não encontrado. Re-execute NN_01_DataGeneration.ipynb."
        )
    Z_train = f['train/z_eq'][:].astype(np.float32)
    Z_val   = f['val/z_eq'][:].astype(np.float32)
    Z_test  = f['test/z_eq'][:].astype(np.float32)

print(f"z_eq carregado: train={Z_train.shape}  val={Z_val.shape}  test={Z_test.shape}")

# Normalização global (escalar único — mesma escala em todas as posições)
z_mu  = float(Z_train.mean())
z_sig = float(Z_train.std()) + 1e-8
Z_train_n = ((Z_train - z_mu) / z_sig).astype(np.float32)
Z_val_n   = ((Z_val   - z_mu) / z_sig).astype(np.float32)
Z_test_n  = ((Z_test  - z_mu) / z_sig).astype(np.float32)

print(f"Scaler global: mean={z_mu:.6f}  std={z_sig:.6f}")
print(f"Z_train_n: mean={Z_train_n.mean():.4f}  std={Z_train_n.std():.4f}")


In [ ]:
# ==============================================================================
# 9. SVM COM z_eq (1024-dim) — Linear e RBF via Nystroem
# ==============================================================================
# Linear SVM via SGD em 1024 dims é escalável (O(N×d) por época).
# RBF via Nystroem aproxima o kernel com n_components=500 features aleatórias.

from sklearn.kernel_approximation import Nystroem

print('SVM Linear (SGD) em z_eq (1024-dim)...')
svm_zeq_lin = Pipeline([
    ('svm', SGDClassifier(loss='hinge', alpha=1e-4, max_iter=100,
                          tol=1e-4, random_state=42, n_jobs=-1,
                          class_weight='balanced', verbose=0))
])
svm_zeq_lin.fit(Z_train_n, y_train)
auc_zeq_lin = roc_auc_score(y_val, svm_zeq_lin.decision_function(Z_val_n))
print(f'  SVM-Linear(z_eq)  val AUC = {auc_zeq_lin:.5f}')

print('\nSVM RBF-Nystroem em z_eq (1024-dim)...')
svm_zeq_rbf = Pipeline([
    ('nystroem', Nystroem(kernel='rbf', gamma=1e-3, n_components=500, random_state=42)),
    ('svm',      SGDClassifier(loss='hinge', alpha=1e-4, max_iter=100,
                               tol=1e-4, random_state=42, n_jobs=-1,
                               class_weight='balanced', verbose=0))
])
svm_zeq_rbf.fit(Z_train_n, y_train)
auc_zeq_rbf = roc_auc_score(y_val, svm_zeq_rbf.decision_function(Z_val_n))
print(f'  SVM-RBF(z_eq)     val AUC = {auc_zeq_rbf:.5f}')

best_svm_zeq = svm_zeq_rbf if auc_zeq_rbf >= auc_zeq_lin else svm_zeq_lin
best_svm_zeq_name = 'SVM-RBF(z_eq)' if auc_zeq_rbf >= auc_zeq_lin else 'SVM-Linear(z_eq)'
print(f'\nMelhor SVM(z_eq): {best_svm_zeq_name}')

# D3F no conjunto de teste
svm_zeq_scores_test = best_svm_zeq.decision_function(Z_test_n)
pd_svm_zeq, _ = eval_d3f(svm_zeq_scores_test, y_test, snr_test)

print('\nSNR  | SVM-4feat | SVM-z_eq')
print('-' * 35)
for s in SNR_BINS:
    sv4 = pd_svm.get(s) or 0
    sz  = pd_svm_zeq.get(s) or 0
    print(f'{s:3d}  | {sv4:8.4f}  | {sz:8.4f}')


In [ ]:
# ==============================================================================
# 10. GMM-LRT COM z_eq — PCA(20) + GMM (1024-dim → impraticável sem redução)
# ==============================================================================
# GMM com covariância full em 1024 dims: matriz 1024×1024 por componente × K × 2.
# PCA(20) preserva as direções de maior variância antes de ajustar as GMMs.

from sklearn.decomposition import PCA

print('PCA(20) em z_eq...')
pca = PCA(n_components=20, random_state=42)
pca.fit(Z_train_n[y_train == 0])   # fit no H0 para capturar variância de ruído

Z_train_pca = pca.transform(Z_train_n).astype(np.float32)
Z_val_pca   = pca.transform(Z_val_n).astype(np.float32)
Z_test_pca  = pca.transform(Z_test_n).astype(np.float32)

var_explained = pca.explained_variance_ratio_.sum()
print(f'Variância explicada (20 PCs): {var_explained:.4f}  ({var_explained*100:.1f}%)')

print('\nTreinando GMM-LRT em z_eq (PCA-20)...')
Zh0 = Z_train_pca[y_train == 0]
Zh1 = Z_train_pca[y_train == 1]

best_bic_zeq = np.inf
best_gmm_zeq_h0 = best_gmm_zeq_h1 = None
best_n_zeq = 1

for n in [1, 2, 4, 8]:
    g0 = GaussianMixture(n_components=n, covariance_type='full',
                         random_state=42, max_iter=200)
    g1 = GaussianMixture(n_components=n, covariance_type='full',
                         random_state=42, max_iter=200)
    g0.fit(Zh0); g1.fit(Zh1)
    bic = g0.bic(Zh0) + g1.bic(Zh1)
    print(f'  GMM(z_eq) n={n}  BIC={bic:.1f}')
    if bic < best_bic_zeq:
        best_bic_zeq, best_n_zeq = bic, n
        best_gmm_zeq_h0, best_gmm_zeq_h1 = g0, g1

print(f'\nMelhor n_components = {best_n_zeq}')

llr_zeq_test = (best_gmm_zeq_h1.score_samples(Z_test_pca)
                - best_gmm_zeq_h0.score_samples(Z_test_pca))
auc_gmm_zeq  = roc_auc_score(y_test, llr_zeq_test)
print(f'GMM-LRT(z_eq PCA-20) test AUC = {auc_gmm_zeq:.5f}')

pd_gmm_zeq, _ = eval_d3f(llr_zeq_test, y_test, snr_test)

print('\nSNR  | GMM-4feat | GMM-z_eq(PCA-20)')
print('-' * 42)
for s in SNR_BINS:
    gm4 = pd_gmm.get(s) or 0
    gz  = pd_gmm_zeq.get(s) or 0
    print(f'{s:3d}  | {gm4:8.4f}  | {gz:16.4f}')


In [ ]:
# ==============================================================================
# 11. GRÁFICO COMPARATIVO — 4 features vs z_eq (1024-dim)
# ==============================================================================

snr_v = np.array(SNR_BINS, dtype=float)
g     = lambda d: np.array([d.get(s) or 0 for s in SNR_BINS])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'SVM e GMM-LRT: 4 features vs z_eq  (α={ALPHA:.0e}, D3F Gaussiano)',
             fontsize=12, fontweight='bold')

# Painel esquerdo: SVM
ax = axes[0]
ax.plot(snr_v, [classical_pd.get(s, 0) for s in SNR_BINS],
        'o-', color='tomato', lw=2.5, ms=7, label='Correlador Clássico')
ax.plot(snr_v, g(pd_svm),     's--', color='steelblue', lw=2, ms=7,
        label=f'SVM 4-feat ({best_svm_name})')
ax.plot(snr_v, g(pd_svm_zeq), 'D-.', color='mediumpurple', lw=2, ms=7,
        label=f'SVM z_eq ({best_svm_zeq_name})')
ax.set(title='SVM', xlabel='SNR (dB)', ylabel='PD', xlim=(-1,31), ylim=(-0.05,1.05))
ax.set_xticks(SNR_BINS); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Painel direito: GMM-LRT
ax = axes[1]
ax.plot(snr_v, [classical_pd.get(s, 0) for s in SNR_BINS],
        'o-', color='tomato', lw=2.5, ms=7, label='Correlador Clássico')
ax.plot(snr_v, g(pd_gmm),     's--', color='darkorange', lw=2, ms=7,
        label=f'GMM-LRT 4-feat (n={best_n})')
ax.plot(snr_v, g(pd_gmm_zeq), 'D-.', color='seagreen', lw=2, ms=7,
        label=f'GMM-LRT z_eq PCA-20 (n={best_n_zeq})')
ax.set(title='GMM-LRT', xlabel='SNR (dB)', ylabel='PD', xlim=(-1,31), ylim=(-0.05,1.05))
ax.set_xticks(SNR_BINS); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = vis_dir / 'NN10_svm_gmm_comparison_1d_vs_4feat.png'
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f'Figura salva → {fig_path}')

# Salvar resultados completos
import joblib
joblib.dump(best_svm_zeq, str(models_dir / 'svm_zeq.pkl'))
joblib.dump(pca,           str(models_dir / 'pca_zeq_20.pkl'))

results_ext = {
    'svm_zeq':  {'model': best_svm_zeq_name,
                 'pd_vs_snr': {str(k): pd_svm_zeq.get(k) for k in SNR_BINS}},
    'gmm_zeq':  {'n_components': best_n_zeq, 'pca_dims': 20,
                 'auc': float(auc_gmm_zeq),
                 'pd_vs_snr': {str(k): pd_gmm_zeq.get(k) for k in SNR_BINS}},
}
out_ext = data_dir / 'nn10_zeq_extension.json'
with open(str(out_ext), 'w') as f:
    json.dump(results_ext, f, indent=2)
print(f'Resultados z_eq salvos → {out_ext}')
